# 15 · Reaction endpoints

`coembed` embeds a reactant and a product as paired conformers: start points for NEB or a TS search.
Each endpoint reads as its own graph, and the pair overlays wherever the graphs agree. No atom map is
needed: the product takes the reactant's atom order from the overlay of the two graphs that keeps the
most bonds.

In [ ]:
import numpy as np
from rdkit import RDLogger

import rxembed as rx

RDLogger.DisableLog("rdApp.*")


def report(reactant, product):
    """Print each pair's all-atom RMSD and every changing bond's length, reactant then product."""
    graphs = [{tuple(sorted((b.GetBeginAtomIdx(), b.GetEndAtomIdx()))) for b in e.mol.GetBonds()} for e in (reactant, product)]
    changing = sorted(graphs[0] ^ graphs[1])
    for cid in reactant.ids:
        r = reactant.mol.GetConformer(cid).GetPositions()
        p = product.mol.GetConformer(cid).GetPositions()
        lengths = "  ".join(f"{i}-{j} {np.linalg.norm(r[i] - r[j]):.2f}/{np.linalg.norm(p[i] - p[j]):.2f}" for i, j in changing)
        print(f"pair {cid}: RMSD {np.sqrt(((r - p) ** 2).sum(1).mean()):.2f} A  {lengths}")

A keto-enol tautomerisation, written without atom maps.

In [ ]:
reactant, product = rx.coembed("CC(=O)C>>C=C(O)C", n=1, seed=1)
report(reactant, product)

A palladium C-H activation, the pincer's phenoxide taking the aryl proton, as two plain SMILES.
`metal=` gives both sides the square-planar sphere; each side must resolve to one isomer.

In [ ]:
cmd_reactant = "COC1=C[CH](->[Pd+2]23<-[N]4=C(C5=[N]->2C(CC2=[N]->3C([O-])=CC=C2)=CC=C5)C=CC=C4)=CC(OC)=C1OC"
cmd_product = "[H]OC1=CC=CC2CC3=CC=CC4C5=[N](->[Pd+2](<-[C-]6=CC(OC)=C(OC)C(OC)=C6)(<-[N]=21)<-[N]=43)C=CC=C5"
reactant, product = rx.coembed((cmd_reactant, cmd_product), metal="SPL", n=3, seed=7)
report(reactant, product)

An H-bonded assembly holds its contacts in both copies. `contacts="auto"` co-embeds one pair per
binding mode found on the reactant, each side an `EnsembleSet` tagged by mode; the acid's H-bond to the
amine holds only in the reactant, since in the product it is the ammonium N-H bond.

In [ ]:
reactants, products = rx.coembed("CC(=O)O.CN>>CC(=O)[O-].C[NH3+]", contacts="auto", n=2, seed=1)
for reactant, product in zip(reactants, products):
    print(reactant.tag["nci"])
    report(reactant, product)